In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()                      # reads your .env file
token = os.getenv("GITHUB_TOKEN")

print("Token loaded:", token is not None)

resp = requests.get(
    "https://api.github.com/rate_limit",
    headers={"Authorization": f"Bearer {token}"},
)
print("Status:", resp.status_code)
print("Requests allowed per hour:", resp.json()["resources"]["core"]["limit"])

Token loaded: True
Status: 200
Requests allowed per hour: 5000


In [5]:
REPO = "pandas-dev/pandas"
headers = {"Authorization": f"Bearer {token}"}

resp = requests.get(
    f"https://api.github.com/repos/{REPO}/issues",
    headers=headers,
    params={"state": "closed", "per_page": 5},
)
issues = resp.json()

for issue in issues:
    print("Number:      ", issue["number"])
    print("Title:       ", issue["title"])
    print("Is a PR?:    ", "pull_request" in issue)
    print("Created:     ", issue["created_at"])
    print("Closed:      ", issue["closed_at"])
    print("Close reason:", issue.get("state_reason"))
    print("Comments:    ", issue["comments"])
    print("Labels:      ", [label["name"] for label in issue["labels"]])
    print("-" * 50)

Number:       71737
Title:        11929 (792×419)
Is a PR?:     False
Created:      2026-10-26T21:54:24Z
Closed:       2026-10-26T22:23:00Z
Close reason: completed
Comments:     0
Labels:       ['Needs Triage']
--------------------------------------------------
Number:       71736
Title:        ENH: pandas to_sql add support for UPSERT operations (INSERT ... ON DUPLICATE KEY UPDATE)
Is a PR?:     False
Created:      2026-10-26T20:45:11Z
Closed:       2026-10-26T21:04:19Z
Close reason: completed
Comments:     1
Labels:       ['Enhancement', 'Needs Triage']
--------------------------------------------------
Number:       71732
Title:        CI: Bump pypa/cibuildwheel to v4.3.4
Is a PR?:     True
Created:      2026-10-26T17:01:29Z
Closed:       2026-10-26T21:26:41Z
Close reason: None
Comments:     2
Labels:       ['Build', 'CI']
--------------------------------------------------
Number:       71727
Title:        Backport PR #71703: CI: Move package-checks.yml to wheel.yml and speed up the

In [ ]:
import time
import pandas as pd
from datetime import datetime, timedelta, timezone

CUTOFF_NEW = datetime.now(timezone.utc) - timedelta(days=180)   # must be at least 6 months old
CUTOFF_OLD = datetime(2024, 1, 1, tzinfo=timezone.utc)            # not older than Jan 2024

rows = []
page = 1

while True:
    resp = requests.get(
        f"https://api.github.com/repos/{REPO}/issues",
        headers=headers,
        params={"state": "all", "sort": "created", "direction": "desc",
                "per_page": 100, "page": page},
    )
        if not isinstance(batch, list):
        print("Stopped by GitHub:", batch.get("message"))
        break
    batch = resp.json()
    if not batch:
        break

    for issue in batch:
        if "pull_request" in issue:
            continue                                   # skip PRs
        created = datetime.fromisoformat(issue["created_at"].replace("Z", "+00:00"))
        if created > CUTOFF_NEW:
            continue                                   # too new, skip
        rows.append({
            "number": issue["number"],
            "title": issue["title"],
            "body": issue["body"] or "",
            "created_at": issue["created_at"],
            "closed_at": issue["closed_at"],
            "state": issue["state"],
            "state_reason": issue.get("state_reason"),
            "author": issue["user"]["login"],
            "author_association": issue["author_association"],
            "labels": [l["name"] for l in issue["labels"]],
            "comments": issue["comments"],
        })

    oldest = datetime.fromisoformat(batch[-1]["created_at"].replace("Z", "+00:00"))
    print(f"page {page}: {len(rows)} issues kept so far, oldest on page {oldest.date()}")
    if oldest < CUTOFF_OLD:
        break
    page += 1
    time.sleep(0.5)                                    # be polite to GitHub's servers

df = pd.DataFrame(rows)
df = df[pd.to_datetime(df["created_at"]) >= CUTOFF_OLD]   # trim anything older than Jan 2024
df.to_json("issues_raw.json", orient="records", lines=True)
print("Saved", len(df), "issues")

page 1: 0 issues kept so far, oldest on page 2026-10-22
page 2: 0 issues kept so far, oldest on page 2026-10-20
page 3: 0 issues kept so far, oldest on page 2026-10-15
page 4: 0 issues kept so far, oldest on page 2026-10-11
page 5: 0 issues kept so far, oldest on page 2026-10-09
page 6: 0 issues kept so far, oldest on page 2026-10-05
page 7: 0 issues kept so far, oldest on page 2026-10-02
page 8: 0 issues kept so far, oldest on page 2026-09-28
page 9: 0 issues kept so far, oldest on page 2026-09-25
page 10: 0 issues kept so far, oldest on page 2026-09-20
page 11: 0 issues kept so far, oldest on page 2026-09-13
page 12: 0 issues kept so far, oldest on page 2026-09-11
page 13: 0 issues kept so far, oldest on page 2026-09-05
page 14: 0 issues kept so far, oldest on page 2026-09-04
page 15: 0 issues kept so far, oldest on page 2026-08-30
page 16: 0 issues kept so far, oldest on page 2026-08-23
page 17: 0 issues kept so far, oldest on page 2026-08-15
page 18: 0 issues kept so far, oldest on

TypeError: string indices must be integers, not 'str'

In [7]:
df = pd.DataFrame(rows)
df.to_json("issues_raw.json", orient="records", lines=True)

print("Saved", len(df), "issues")
print("From", df["created_at"].min()[:10], "to", df["created_at"].max()[:10])
print(df["state_reason"].value_counts(dropna=False))

Saved 1920 issues
From 2024-05-01 to 2026-04-29
state_reason
completed      1557
NaN             296
not_planned      49
reopened         10
duplicate         8
Name: count, dtype: int64


In [8]:
df = pd.read_json("issues_raw.json", lines=True)

df["created_at"] = pd.to_datetime(df["created_at"])
df["closed_at"] = pd.to_datetime(df["closed_at"])

# How many days it took to close (empty if still open)
df["days_to_close"] = (df["closed_at"] - df["created_at"]).dt.days

# Rule 2: keep issues still open on day 7
#   → either it was never closed, OR it closed after day 7
still_open_day7 = df["closed_at"].isna() | (df["days_to_close"] > 7)
df = df[still_open_day7]

# Rule 4: label = 1 if fixed, 0 if not
#   → fixed means: currently closed AND close reason is "completed"
df["fixed"] = ((df["state"] == "closed") & (df["state_reason"] == "completed")).astype(int)

print("Issues left:", len(df))
print(df["fixed"].value_counts())

Issues left: 1212
fixed
1    874
0    338
Name: count, dtype: int64


In [10]:
from tqdm.auto import tqdm   # progress bar; if it errors, run: pip install tqdm

def first_week_comments(issue_number, created_at):
    resp = requests.get(
        f"https://api.github.com/repos/{REPO}/issues/{issue_number}/comments",
        headers=headers,
        params={"per_page": 100},
    )
    comments = resp.json()
    if not isinstance(comments, list):
        return []
    cutoff = created_at + pd.Timedelta(days=7)
    keep = []
    for c in comments:
        c_time = pd.to_datetime(c["created_at"])
        if c_time <= cutoff:                       # only the first 7 days
            keep.append({
                "author": c["user"]["login"],
                "author_association": c["author_association"],
                "created_at": c["created_at"],
                "body": c["body"] or "",
            })
    return keep

all_comments = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    all_comments.append(first_week_comments(row["number"], row["created_at"]))
    time.sleep(0.2)

df["week1_comments"] = all_comments
df.to_json("issues_week1.json", orient="records", lines=True, date_format="iso")
print("Saved", len(df), "issues with first-week comments")

  0%|          | 0/1212 [00:00<?, ?it/s]

Saved 1212 issues with first-week comments


In [11]:
df["n_week1_comments"] = df["week1_comments"].apply(len)

print("Issues with at least one first-week comment:", (df["n_week1_comments"] > 0).sum())
print()
print("Who comments? (author_association counts)")
assoc = [c["author_association"] for comments in df["week1_comments"] for c in comments]
print(pd.Series(assoc).value_counts())

Issues with at least one first-week comment: 988

Who comments? (author_association counts)
MEMBER         1580
CONTRIBUTOR     999
NONE            892
Name: count, dtype: int64
